# 🧪 Lab 1: OLAP, Stockage en Colonnes et BigQuery Console UI

Dans ce lab pratique, vous allez directement utiliser l'interface web **BigQuery Console** (Studio) sur Google Cloud Platform (`console.cloud.google.com/bigquery`).

Nous allons travailler sur le jeu de données e-commerce de référence de Google : **TheLook eCommerce** (`bigquery-public-data.thelook_ecommerce`), exactement celui présenté dans le cours théorique !

L'objectif est d'observer et de mesurer la différence fondamentale d'une base analytique (**OLAP**) : le **stockage en colonnes** et le modèle de facturation associé (**bytes scanned / octets traités**).

### 🎯 Objectifs :
1. Ouvrir l'éditeur de requêtes SQL dans la console BigQuery sur le projet `lasalle-big-data`.
2. **(Démo Formateur)** Rechercher et mettre en favori (épingler / Star) le jeu de données public `bigquery-public-data.thelook_ecommerce` dans le volet Explorateur.
3. Observer l'indicateur d'audit (**Dry Run automatique**) en haut/bas de l'éditeur SQL : *« Cette requête traitera X Mo/Go lors de son exécution »*.
4. Mesurer l'impact financier et technique du stockage en colonnes (`SELECT *` vs `SELECT colonnes_spécifiques`) sur la table d'événements de clics `events`.
5. Analyser les détails d'exécution (panneau d'information d'exécution, parallélisme des slots, timeline).
6. Interroger les commandes et paniers clients (`orders` & `order_items`).

---

## 0. Accès à la Console BigQuery & Démo Datasets Publics

1. Rendez-vous sur votre navigateur à l'adresse suivante :
   👉 **[https://console.cloud.google.com/bigquery?project=lasalle-big-data](https://console.cloud.google.com/bigquery?project=lasalle-big-data)**
2. Vérifiez en haut à gauche que le projet sélectionné est bien **`lasalle-big-data`**.
3. Cliquez sur le bouton **`+ Saisir une nouvelle requête`** (ou l'onglet éditeur SQL vide).

> 💡 **Astuce sur le Dry Run dans l'UI** :
> Dès que vous collez une requête SQL valide dans l'éditeur, une **coche verte** s'affiche avec le texte :
> `Cette requête traitera X Mo (ou Go) lors de son exécution.`
> **BigQuery calcule cela avant même que vous ne cliquiez sur Exécuter (Run) ! C'est gratuit et instantané.**

---

### 👨‍🏫 Démonstration Formateur : Récupérer et Mettre en Favori un Dataset Public (`bigquery-public-data`)

Avant de commencer la rédaction des requêtes SQL, le formateur montre comment rechercher et ajouter en **favori (étoile ⭐️ / Star / Épingler)** un jeu de données public dans la console BigQuery.

1. **Accéder au panneau d'ajout de données** :
   - Dans le volet de gauche **Explorateur** (Explorer), cliquez sur le bouton **`+ AJOUTER`** (Add Data) situé tout en haut.
   - Choisissez l'option **`Parcourir les jeux de données publics`** (Browse public datasets) ou **`Épingler un projet par son nom`** (Star a project by name).

2. **Rechercher le jeu de données public** :
   - Tapez **`thelook_ecommerce`** dans le catalogue public GCP ou entrez le nom exact du projet public : **`bigquery-public-data`**.
   - Cliquez sur le jeu de données **TheLook eCommerce** puis sur **`Afficher le jeu de données`**.

3. **Mettre en favori (Star / Épingler)** :
   - Observez le projet **`bigquery-public-data`** qui apparaît désormais dans votre volet **Explorateur** (sous votre projet principal `lasalle-big-data`).
   - Cliquez sur l'icône **Étoile ⭐️ (Mettre en favori / Star)** à côté du dataset `thelook_ecommerce` ou du projet `bigquery-public-data`.
   - 📌 **Résultat** : Le dataset public reste enregistré dans votre panneau Explorateur pour tous vos accès futurs sans devoir saisir à chaque fois le chemin complet !

---
## 1. Démonstration du Stockage en Colonnes sur TheLook Events

La table `bigquery-public-data.thelook_ecommerce.events` enregistre l'ensemble des clics, vues de produits et ajouts au panier de tous les utilisateurs d'un site e-commerce fictif.

Dans BigQuery (**OLAP**), les données sont stockées **colonne par colonne** (moteur Capacitor).
Si une table fait plusieurs centaines de Mo/Go et que vous ne demandez que 2 colonnes, BigQuery **n'ouvre physiquement que ces 2 colonnes** !

### Étape 1.1 : La mauvaise pratique — `SELECT *`

Copiez et collez la requête suivante dans l'éditeur BigQuery, **sans cliquer sur Exécuter** :

```sql
-- ⚠️ MAUVAISE PRATIQUE SUR BIGQUERY : SELECT * scanne TOUTES les 13 colonnes du disque
SELECT *
FROM `bigquery-public-data.thelook_ecommerce.events`
WHERE event_type = 'product'
LIMIT 10;
```

🔍 **Observation attendue :**
- Regardez la coche verte du validateur : le volume affiché est d'environ **~366 Mo** même pour seulement 10 lignes !
- ⚠️ **Règle d'or BigQuery** : `LIMIT 10` ne réduit **JAMAIS** le coût d'une requête, car BigQuery doit d'abord scanner l'ensemble des colonnes sur disque avant de filtrer les 10 premières lignes.

### Étape 1.2 : La bonne pratique analytique — Sélection ciblée des colonnes

Remplacez maintenant le code dans l'éditeur par cette requête optimisée qui ne sélectionne que les 2 colonnes utiles (`event_type` et `traffic_source`) :

```sql
-- ✅ BONNE PRATIQUE : Ne lire que les colonnes nécessaires (event_type, traffic_source)
SELECT 
    traffic_source,
    COUNT(1) AS total_events
FROM `bigquery-public-data.thelook_ecommerce.events`
WHERE event_type = 'product'
GROUP BY traffic_source
ORDER BY total_events DESC;
```

🔍 **Observation & Exécution :**
1. Regardez à nouveau la coche verte : le volume scanné s'est effondré à seulement **~39 Mo** (**réduction de près de 90%** des octets traités !).
2. Cliquez sur le bouton bleu **`▷ Exécuter` (Run)**.
3. Observez le résultat : quelle source de trafic (Search, Organic, Email, Facebook) génère le plus de consultations de fiches produits ?
4. Cliquez sur l'onglet **Informations sur l'exécution** (Execution details) sous la requête pour observer les étapes du calcul distribué.

---
## 2. Analyse Métier : Ventes & Statuts des Commandes TheLook

Interrogeons la table des articles commandés (`order_items`).
Nous souhaitons analyser le chiffre d'affaires, le panier moyen et le volume d'articles vendus selon le statut de la commande (Complete, Shipped, Returned, Cancelled).

Collez cette requête SQL dans un nouvel onglet de requête :

```sql
-- Analyse financière des ventes et retours sur TheLook
SELECT 
    status,
    COUNT(1)                     AS total_items_ordered,
    ROUND(AVG(sale_price), 2)    AS avg_item_price_usd,
    ROUND(SUM(sale_price), 2)    AS total_revenue_usd
FROM `bigquery-public-data.thelook_ecommerce.order_items`
GROUP BY status
ORDER BY total_revenue_usd DESC;
```

### 📊 Décryptage des Métriques d'Exécution dans la Console

Une fois la requête exécutée, regardez en bas dans les onglets de résultats :

1. **Onglet « Résultats de la requête » (Query Results)** :
   - Analysez la part des commandes retournées (`Returned`) ou annulées (`Cancelled`) par rapport aux commandes livrées (`Complete`).
2. **Onglet « Informations sur l'exécution » (Job Information / Execution details)** :
   - **Octets traités (Bytes billed / Bytes processed)** : Le volume exact lu sur le stockage distribué Colossus.
   - **Temps écoulé (Elapsed time)** : Le temps réel d'attente (wall clock, ex: ~0.5 seconde).
   - **Temps processeur total (Slot time consumed)** : Le temps cumulé de tous les cœurs CPU mobilisés en parallèle.
   - 👉 **Rapport Slot Time / Elapsed Time** = Degré de parallélisation (ex: 2.5s / 0.5s $\approx$ **5 processeurs ont travaillé en parallèle pour vous !**).

---
## 3. Optionnel : Vérification en Ligne de Commande (Cloud Shell / `bq`)

Si vous ouvrez le **Cloud Shell** (icône `>_` tout en haut à droite de la console GCP), vous pouvez aussi lancer le Dry Run en ligne de commande avec le CLI officiel `bq` :

```bash
bq query \
  --use_legacy_sql=false \
  --dry_run \
  "SELECT status, COUNT(1) FROM \`bigquery-public-data.thelook_ecommerce.order_items\` GROUP BY 1"
```

Le terminal vous répondra instantanément sans facturation ni exécution :
`Query successfully validated. Assuming the tables are not modified, running this query will process XXXXX bytes of data.`

---
## 4. Exercice Pratique : À vous de jouer ! 🏋️

### Énoncé :
Sur la table des produits `bigquery-public-data.thelook_ecommerce.products` :

1. Rédigez une requête SQL qui affiche pour chaque catégorie de produit (`category`) :
   - Le nombre total de produits référencés 
   - Le prix de vente moyen conseillé
   - Le coût moyen de fabrication
   - La marge unitaire moyenne estimée

2. Triez par la marge moyenne décroissante.

3. **Avant de cliquer sur Exécuter** : notez le volume de données estimé par la coche verte.

4. Exécutez la requête et identifiez quelle catégorie de vêtements est la plus rentable pour TheLook !

---
## ✅ Checkpoint de Validation du Module

- [ ] J'ai accédé à la console BigQuery sur le projet `lasalle-big-data`.
- [ ] J'ai suivi la démo formateur pour rechercher et mettre en favori (star/épingler) un dataset public (`bigquery-public-data.thelook_ecommerce`).
- [ ] J'ai compris le principe du stockage en colonnes (Capacitor) sur la table `thelook_ecommerce.events` et pourquoi `SELECT *` est proscrit.
- [ ] J'ai repéré l'indicateur d'audit (**Dry Run**) dans la console BigQuery qui estime gratuitement le volume avant exécution.
- [ ] J'ai compris que `LIMIT` ne réduit pas les octets scannés sur BigQuery.
- [ ] J'ai exploré l'onglet *Informations sur l'exécution* (Temps réel vs Slot Time / parallélisme) sur `thelook_ecommerce.order_items`.